In [6]:
import sys
sys.version

'3.10.12 (main, Jun 11 2023, 05:26:28) [GCC 11.4.0]'

In [7]:
!pip install python-mecab-ko

In [8]:
from mecab import MeCab
mecab = MeCab()

In [4]:
mecab.morphs('영등포구청역에 있는 맛집 좀 알려주세요.')

['영등포구청역', '에', '있', '는', '맛집', '좀', '알려', '주', '세요', '.']

In [ ]:
mecab.nouns("헤이 테크 블로그입니다.")

In [ ]:
!pip install pyLDAvis

In [10]:
import numpy as np
import pandas as pd
import warnings # 경고 메시지 무시
warnings.filterwarnings(action='ignore')
# 한국어 형태소 분석기 중 성능이 가장 우수한 Mecab 사용
from tqdm import tqdm # 작업 프로세스 시각화
import re # 문자열 처리를 위한 정규표현식 패키지
import gensim # LDA 모델 활용 목적
from gensim import corpora # 단어 빈도수 계산 패키지
import pyLDAvis.gensim_models # LDA 시각화용 패키지
from collections import Counter # 단어 등장 횟수 카운트
import pickle
import csv
import os

In [11]:
import pickle
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis
from gensim.models.coherencemodel import CoherenceModel
import matplotlib.pyplot as plt

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [12]:
from gensim.models.ldamodel import LdaModel
from gensim.models.callbacks import CoherenceMetric
from gensim import corpora
from gensim.models.callbacks import PerplexityMetric

import logging
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [50]:
df = pd.read_csv('/content/sample_data/최종_it.csv')
df

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,회사이름,직무,면접질문,면접느낌,합불
0,(주)티몬,유통////////무역,"자기소개, 어떻게 가격을 협의할 건지, 성향",도전적인 성향을 좋아하는 것 같았음,합격
1,(주)티몬,영업////////제휴,어떤 브랜드사 파트너가 담당했었는지 다른 카테고리 담당하게 되어도 괜찮은지 조직에 ...,무난한 면접이였고 1:1방식의 면접으로 진행되었다,합격
2,(주)티몬,미디어////////홍보,티몬에 대해 알고있는 것 모두 기획하고 싶은 방송이 있는지,네이버에 티몬 검색해서 나온거 모두 말함 최근 이슈는 언론 기사 활용,합격
3,(주)티몬,영업////////제휴,기억남는건 없어요 진짜 일반 적인 질문들이엿고 이정도 질문을 못할 정도면 다른 회사...,전부다 차분히 좋게 말해주셧고 면접분위기는 엄청 편한? 하하호호 분위기엿어요,합격
4,(주)티몬,마케팅////////시장조사,다른 분야의 일을 하다가 왜 어쩌다 이 직무로 지원했는가,학교 수업과 동아리로 들었던 샵마스터 수업과 네이버 스마트 스토어 운영해봤던 경험을...,합격
...,...,...,...,...,...
61790,소리바다(주),개발,난해한 질문들을 요구했고 업무와 크게 관련되지 않은듯한 질문의 연속이였음.,NaN,불합격
61791,소리바다(주),개발,별도로 어려운 질문보다는 각오나 의견을 묻는 정도,NaN,합격
61792,소리바다(주),개발,버블정렬 알고리즘 말해보라고 했는데 당황해서 딴소리 했네요 ㅋㅋ,피보나치수열을 말해버렸다고 들었네요 ㅋㅋ,합격
61793,소리바다(주),디자인,남자친구 있어요?,NaN,합격


#### 직무별 면접질문 LDA

In [51]:
career = df['직무'].unique()
print(df['직무'].value_counts())
print(len(career))

개발                  27093
기획////////경영         9612
영업////////제휴         5145
마케팅////////시장조사      5072
디자인                  3625
서비스////////고객지원      3275
인사////////총무         1532
미디어////////홍보        1208
데이터                  1163
금융////////재무          941
유통////////무역          542
교육                    537
엔지니어링                 357
연구개발                  349
생산관리////////품질관리      347
기타                    329
전문직                   264
생산////////제조          209
법률////////법무          146
특수계층////////공공         36
의약                     11
|                       2
Name: 직무, dtype: int64
22


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [52]:
value_counts = df['직무'].value_counts()
filtered_values = value_counts[value_counts >= 5]

career_list = filtered_values.index.tolist()
print(career_list)
print(len(career_list))

['개발', '기획////////경영', '영업////////제휴', '마케팅////////시장조사', '디자인', '서비스////////고객지원', '인사////////총무', '미디어////////홍보', '데이터', '금융////////재무', '유통////////무역', '교육', '엔지니어링', '연구개발', '생산관리////////품질관리', '기타', '전문직', '생산////////제조', '법률////////법무', '특수계층////////공공', '의약']
21


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [53]:
words_to_remove = ['이유','사항','대로','Why','you','건지','to','nan','1.','생각','무엇','기타','한국전력','대답','아모레','내용','and','자신','확인','이전','소개서','why','퍼시픽','for','위주','What','한전','at','관련','자소','응대','꼬리','벅스','기반','think','소개','own','당신','자기소개','등등','필요','스타','do','서비스','the','면접','in','가스공사','경우','what','동안','얘기','질문','고객','설명','이력서','자기','정도','여기','의료','소서','텐데','IT','지원','본인','동기', '가스안전공사','대부분','your','의약', '우리','//']

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [54]:
for i in range(0, len(career_list)):
  df_que = df.loc[df['직무'] == career_list[i]]['면접질문']

  # que_clean = [re.sub(r'[^ ㄱ-ㅣ가-힣]+'," ", str(doc))  for doc in df_que]

  word_list=[]

  interview_prop = []
  interview_prep = []

  for sentence in df_que:
      # english_tokens = sentence.split()

      korea_tokens = []

      for token in re.split(r'([a-zA-Z]+)', str(sentence)):
          if token:
              if re.match(r'[a-zA-Z]+', token):
                  korea_tokens.append(token)
              else:
                  korea_tokens.extend(mecab.nouns(token))

      combined_tokens = korea_tokens

      interview_prop.append(list(combined_tokens))

  for k in interview_prop:
    jj = []
    for j in k:
        if len(j) > 1:
            if j not in words_to_remove:
                jj.append(j)

    interview_prep.append(list(jj))

  # for k in range(len(que_clean)):
  #   nouns = mecab.nouns(que_clean[k])
  #   nouns = [word for word in nouns if len(word) > 1]
  #   nouns_2 = [word for word in nouns if word not in words_to_remove]
  #   word_list.append(nouns_2)

  dictionary = corpora.Dictionary(interview_prep)
  dictionary.filter_extremes(no_below=2, no_above=0.5)
  corpus = [dictionary.doc2bow(text) for text in interview_prep]

  max = 0
  count = 0
  chunksize = 2000
  passes = 15
  iterations = 400
  eval_every = None

  temp = dictionary[0]
  id2word = dictionary.id2token

  for p in range(5, 11):
      ldaModel = LdaModel(
        corpus=corpus,
        id2word=id2word,
        chunksize=chunksize,
        alpha='auto',
        eta='auto',
        iterations=iterations,
        num_topics=p,
        passes=passes,
        eval_every=eval_every
        )
      coherence_model_lda=CoherenceModel(model= ldaModel, texts=interview_prep,
                                        dictionary=dictionary, topn=5)
      coherence_lda=coherence_model_lda.get_coherence()
      print('coherence: %.4f.' % coherence_lda)
      if coherence_lda > max:
          max = coherence_lda
          count = p

  print(max)
  print(count)

  num_topics = count
  chunksize = 2000
  passes = 15
  iterations = 400
  eval_every = None

  temp = dictionary[0]
  id2word = dictionary.id2token

  model = LdaModel(
      corpus=corpus,
      id2word=id2word,
      chunksize=chunksize,
      alpha='auto',
      eta='auto',
      iterations=iterations,
      num_topics=num_topics,
      passes=passes,
      eval_every=eval_every
      )

  # all_topic_words = []

  # for topic_id in range(num_topics):
  #     topic_words = model.show_topic(topic_id, topn=model.num_terms)
  #     topic_words = [word for word, prob in topic_words]
  #     all_topic_words.append(topic_words[:5])

  # result_df = pd.DataFrame({'Topic_Words': all_topic_words})
#   result_df = pd.DataFrame(all_topic_words)


  kk = pd.DataFrame(model.print_topics(num_words=5))
  kk['직무'] = career_list[i]

  print(kk)

  if not os.path.exists('it_lda.csv'):
    kk.to_csv('it_lda.csv', index=False, mode='w', encoding = 'cp949')
  else:
    kk.to_csv('it_lda.csv', index=False, mode='a', encoding = 'cp949', header=False)


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


coherence: 0.6397.
coherence: 0.5967.
coherence: 0.6098.
coherence: 0.6130.
coherence: 0.6108.
coherence: 0.6045.
0.6396516600586521
5
   0                                                  1  직무
0  0  0.103*"업무" + 0.034*"코딩" + 0.032*"테스트" + 0.031*...  개발
1  1  0.058*"성격" + 0.056*"장단점" + 0.056*"영어" + 0.025*...  개발
2  2  0.175*"회사" + 0.052*"가능" + 0.041*"입사" + 0.028*"...  개발
3  3  0.060*"문제" + 0.046*"사용" + 0.042*"기본" + 0.035*"...  개발
4  4  0.078*"프로젝트" + 0.060*"기술" + 0.048*"경험" + 0.033...  개발
coherence: 0.5330.
coherence: 0.6330.
coherence: 0.5727.
coherence: 0.6011.
coherence: 0.6173.
coherence: 0.5698.
0.6330265318237774
6
   0                                                  1            직무
0  0  0.071*"영어" + 0.029*"가지" + 0.024*"분위기" + 0.022*...  기획////////경영
1  1  0.036*"방법" + 0.032*"해결" + 0.031*"상황" + 0.024*"...  기획////////경영
2  2  0.122*"게임" + 0.024*"기획" + 0.022*"개선" + 0.020*"...  기획////////경영
3  3  0.071*"업무" + 0.064*"프로젝트" + 0.057*"진행" + 0.056...  기획////////경영
4  4  0.073*"기억" + 0

#### 합불에 따른 면접느낌에 대한 LDA

In [48]:
words_to_remove = ['이유','사항','때문','4.','2.','대로','Why','////','////////','you','...','..','.','건지','.\r','to','nan','1.','3.','생각','무엇','기타','한국전력','대답','아모레','내용','and','자신','확인','이전','소개서','why','퍼시픽','for','위주','What','한전','at','관련','자소','응대','꼬리','벅스','기반','think','소개','own','당신','자기소개','등등','필요','스타','do','서비스','the','면접','in','가스공사','경우','what','동안','얘기','질문','고객','설명','이력서','자기','정도','여기','의료','소서','텐데','IT','지원','본인','동기', '가스안전공사','대부분','your','의약', '우리','//']

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [49]:
PorF_list = ['합격', '불합격']

for i in range(0, len(PorF_list)):
  df_que = df.loc[df['합불'] == PorF_list[i]]['면접느낌']

  # que_clean = [re.sub(r'[^ ㄱ-ㅣ가-힣]+'," ", str(doc))  for doc in df_que]

  word_list=[]

  interview_prop = []
  interview_prep = []

  for sentence in df_que:
      # english_tokens = sentence.split()

      korea_tokens = []

      for token in re.split(r'([^ ㄱ-ㅣ가-힣]+)', str(sentence)):
          if token:
              if re.match(r'[^ ㄱ-ㅣ가-힣]+', token):
                  korea_tokens.append(token)
              else:
                  korea_tokens.extend(mecab.nouns(token))

      combined_tokens = korea_tokens

      interview_prop.append(list(combined_tokens))

  for k in interview_prop:
    jj = []
    for j in k:
        if len(j) > 1:
            if j not in words_to_remove:
                jj.append(j)

    interview_prep.append(list(jj))

  # for k in range(len(que_clean)):
  #   nouns = mecab.nouns(que_clean[k])
  #   nouns = [word for word in nouns if len(word) > 1]
  #   nouns_2 = [word for word in nouns if word not in words_to_remove]
  #   word_list.append(nouns_2)

  dictionary = corpora.Dictionary(interview_prep)
  dictionary.filter_extremes(no_below=2, no_above=0.5)
  corpus = [dictionary.doc2bow(text) for text in interview_prep]

  max = 0
  count = 0
  chunksize = 2000
  passes = 15
  iterations = 400
  eval_every = None

  temp = dictionary[0]
  id2word = dictionary.id2token

  for p in range(5, 11):
      ldaModel = LdaModel(
        corpus=corpus,
        id2word=id2word,
        chunksize=chunksize,
        alpha='auto',
        eta='auto',
        iterations=iterations,
        num_topics=p,
        passes=passes,
        eval_every=eval_every
        )
      coherence_model_lda=CoherenceModel(model= ldaModel, texts=interview_prep,
                                        dictionary=dictionary, topn=5)
      coherence_lda=coherence_model_lda.get_coherence()
      print('coherence: %.4f.' % coherence_lda)
      if coherence_lda > max:
          max = coherence_lda
          count = p

  print(max)
  print(count)

  num_topics = count
  chunksize = 2000
  passes = 15
  iterations = 400
  eval_every = None

  temp = dictionary[0]
  id2word = dictionary.id2token

  model = LdaModel(
      corpus=corpus,
      id2word=id2word,
      chunksize=chunksize,
      alpha='auto',
      eta='auto',
      iterations=iterations,
      num_topics=num_topics,
      passes=passes,
      eval_every=eval_every
      )

  # all_topic_words = []

  # for topic_id in range(num_topics):
  #     topic_words = model.show_topic(topic_id, topn=model.num_terms)
  #     topic_words = [word for word, prob in topic_words]
  #     all_topic_words.append(topic_words[:5])

  # result_df = pd.DataFrame({'Topic_Words': all_topic_words})
#   result_df = pd.DataFrame(all_topic_words)


  kk = pd.DataFrame(model.print_topics(num_words=5))
  kk['합불'] = PorF_list[i]

  print(kk)

  if not os.path.exists('itfeel_lda.csv'):
    kk.to_csv('itfeel_lda.csv', index=False, mode='w', encoding = 'cp949')
  else:
    kk.to_csv('itfeel_lda.csv', index=False, mode='a', encoding = 'cp949', header=False)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


coherence: 0.6667.
coherence: 0.6025.
coherence: 0.6272.
coherence: 0.6474.
coherence: 0.6154.
coherence: 0.6286.
0.6667066614048072
5
   0                                                  1  합불
0  0  0.127*"분위기" + 0.085*"진행" + 0.056*"편안" + 0.054*...  합격
1  1  0.051*"시간" + 0.033*"임원" + 0.026*"전공" + 0.024*"...  합격
2  2  0.097*"답변" + 0.040*"회사" + 0.035*"경험" + 0.034*"...  합격
3  3  0.051*"문제" + 0.049*"기억" + 0.047*"합격" + 0.021*"...  합격
4  4  0.077*"사람" + 0.027*"게임" + 0.023*"공부" + 0.021*"...  합격
coherence: 0.6644.
coherence: 0.6814.
coherence: 0.7066.
coherence: 0.6776.
coherence: 0.6530.
coherence: 0.7054.
0.7066377059504473
7
   0                                                  1   합불
0  0  0.047*"어필" + 0.031*"중요" + 0.023*"성격" + 0.023*"...  불합격
1  1  0.095*"분위기" + 0.090*"진행" + 0.056*"느낌" + 0.056*...  불합격
2  2  0.024*"긍정" + 0.021*"노력" + 0.019*"기업" + 0.017*"...  불합격
3  3  0.050*"사람" + 0.036*"회사" + 0.023*"실무" + 0.022*"...  불합격
4  4  0.070*"문제" + 0.061*"게임" + 0.042*"인성" + 0.030*"...  불합격
5  5